In [ ]:
import numpy as np
from dataclasses import dataclass
import matplotlib.pyplot as plt
import scipy.optimize
import copy

# Needed for animations
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Needed for interactive slider bars
import ipywidgets


In [ ]:
@dataclass
class Parameters:
    Nx: int
    Ny: int
    Nz: int
    a: float
    sigma3: float
    m3sq: float
    g3: float
    lambda3: float
    rng: np.random.Generator

In [ ]:
def local_potential(phi: float, p: Parameters):
    return p.sigma3*phi + (1.0/2.0)*p.m3sq*phi**2 + (1.0/6.0)*p.g3*phi**3 + (1.0/24.0)*p.lambda3*phi**4

def laplacian(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0

    total += lattice[(x + 1) % p.Nx][y][z]
    total += lattice[x][(y + 1) % p.Ny][z]
    total += lattice[x][y][(z + 1) % p.Nz]
    total += lattice[(x + p.Nx - 1) % p.Nx][y][z]
    total += lattice[x][(y + p.Ny - 1) % p.Ny][z]
    total += lattice[x][y][(z + p.Nz - 1) % p.Nz]

    total -= 6.0 * lattice[x][y][z]

    return total

def laplacian_contribution(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0
    
    total += 2.0 * lattice[(x + 1) % p.Nx][y][z]
    total += 2.0 * lattice[x][(y + 1) % p.Ny][z]
    total += 2.0 * lattice[x][y][(z + 1) % p.Nz]
    total += 2.0 * lattice[(x + p.Nx - 1) % p.Nx][y][z]
    total += 2.0 * lattice[x][(y + p.Ny - 1) % p.Ny][z]
    total += 2.0 * lattice[x][y][(z + p.Nz - 1) % p.Nz]

    total -= 6.0 * lattice[x][y][z]

    return total

def lattice_contribution(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    total = 0.0

    total += (-1.0/(2.0*p.a**2))*lattice[x][y][z]*laplacian_contribution(lattice, x, y, z, p)
    total += local_potential(lattice[x][y][z], p)

    return total

def potential(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):                
                total += local_potential(lattice[x][y][z], p)

    return total   

def lattice_action(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):                
                total += (-1.0/(2.0*p.a**2))*lattice[x][y][z]*laplacian(lattice, x, y, z, p)
                total += local_potential(lattice[x][y][z], p)

    return total


In [ ]:
def init_lattice_cold(p: Parameters):
    lattice = np.zeros(shape=(p.Nx,p.Ny,p.Nz), dtype=float)
    return lattice

def init_lattice_warm(p: Parameters, seed=1):

    rng = np.random.default_rng(seed=seed)
    lattice = rng.uniform(-1, 1, size=(p.Nx,p.Ny,p.Nz))
    return lattice

In [ ]:
def metropolis_update(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):
    original = lattice[x][y][z]

    before = lattice_contribution(lattice, x,y,z, p)  

    lattice[x][y][z] += p.rng.uniform(-1, 1)

    after = lattice_contribution(lattice, x,y,z, p)

    if after < before or np.exp(-1.0*(after-before)) > p.rng.uniform(0,1):
        # print(after-before, 'accept')
        return 1, (after-before)

    # print(after-before, 'reject')
    lattice[x][y][z] = original
    return 0, 0.0
    
def overrelax_update(lattice: np.ndarray, x: int, y: int, z: int, p: Parameters):

    phi_0 = lattice[x][y][z]

    alpha = (1.0/24.0)*p.lambda3
    beta = (1.0/6.0)*p.g3
    gamma = (-1.0/(2.0*p.a**2)) * -6.0 + (1.0/2.0)*p.m3sq
    delta = 2.0 * (-1.0/(2.0*p.a**2)) * (laplacian(lattice, x, y, z, p) + 6.0 * phi_0) + p.sigma3

    
    #   this is equivalent to solving
    #        alpha*(phi-phi0)*(phi**3 + a*phi**2 + b*phi + d) = 0
    #    where
    #        alpha = phi0
    #        beta = b/a + phi0**2
    #        gamma = d/a + alpha*beta
   
    a = phi_0 + beta / alpha
    b = phi_0 * a + gamma / alpha
    d = phi_0 * b + delta / alpha

  
    #    solving
    #        phi**3 + a*phi**2 + b*phi + d = 0

    def poly(x):
        return x**3 + a*x**2 + b*x + d
    #X = np.linspace(-10,10,500)
    #plt.plot(X, poly(X))
    roots = scipy.optimize.fsolve(poly, 0.1)

    # print('Roots:', roots)

    phi = roots[0]

    if len(roots) == 3:
        prob = p.rng.uniform(0, 1)

        if prob > 2.0/3.0:
            phi = roots[2]
        elif prob > 1.0/3.0:
            phi = roots[1]

    measure = phi_0 * (4.0 * alpha * phi_0 * phi_0 + 3.0 * beta * phi_0 + 2.0 * gamma) + delta
    measure /= phi * (4.0 * alpha * phi * phi + 3.0 * beta * phi + 2.0 * gamma) + delta
    measure = np.log(np.abs(measure))

    lattice[x][y][z] = phi

    if measure > np.log(1.0) or measure > np.log(p.rng.uniform(0, 1)):
        return 1
    
    lattice[x][y][z] = phi_0
    return 0

In [ ]:
def sweep(lattice: np.ndarray, p: Parameters):
    total_change = 0.0
 
    updated = 0
    accepted = 0
    before = lattice_action(lattice, p)

    for evenodd in (0,1):
        for x in range(p.Nx):
            for y in range(p.Ny):
                for z in range(p.Nz):
                    if (x + y + z) % 2 == evenodd:
                        accept, change = metropolis_update(lattice, x, y, z, p)
                        total_change += change
                        updated += 1
                        accepted += accept
    
    after = lattice_action(lattice, p)
    # print(before, after, total_change, after-before, (after-before)-total_change)
    return updated, accepted

def sweep_overrelax(lattice: np.ndarray, p: Parameters):
    total_change = 0.0
 
    updated = 0
    accepted = 0

    for evenodd in (0,1):
        for x in range(p.Nx):
            for y in range(p.Ny):
                for z in range(p.Nz):
                    if (x + y + z) % 2 == evenodd:
                        accept= overrelax_update(lattice, x, y, z, p)
                        updated += 1
                        accepted += accept
    
    after = lattice_action(lattice, p)
    # print(before, after, total_change, after-before, (after-before)-total_change)
    return updated, accepted

In [ ]:
def avg_phi(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += lattice[x][y][z]

    return total/(p.Nx*p.Ny*p.Nz)

def avg_phi_sq(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += lattice[x][y][z]**2

    return total/(p.Nx*p.Ny*p.Nz)

def avg_phi_3(lattice: np.ndarray, p: Parameters):
    total = 0.0

    for x in range(p.Nx):
        for y in range(p.Ny):
            for z in range(p.Nz):
                total += lattice[x][y][z]**3

    return total/(p.Nx*p.Ny*p.Nz)


In [ ]:
def run(sweeps: int, lattice: np.ndarray, p: Parameters, save_slices=False):

    total_updated = 0
    total_accepted = 0

    total_overrelax_updated = 0
    total_overrelax_accepted = 0
    results = []
    if save_slices:
        slices = []

    for i in range(sweeps):
        for j in range(8):
            updated, accepted = sweep(lattice, p)
            total_updated += updated
            total_accepted += accepted

        updated, accepted = sweep_overrelax(lattice, p)
        total_overrelax_updated += updated
        total_overrelax_accepted += accepted        
        results.append((avg_phi(lattice, p),avg_phi_sq(lattice,p),avg_phi_3(lattice,p),lattice_action(lattice,p),potential(lattice,p)))

        if save_slices:
            slices.append(copy.deepcopy(lattice[0,:]))

    print('Metropolis: %d sites updated, %d accepted, %g rate' % (total_updated, total_accepted, float(total_accepted)/float(total_updated)))
    print('Overrelax: %d sites updated, %d accepted, %g rate' % (total_overrelax_updated, total_overrelax_accepted, float(total_overrelax_accepted)/float(total_overrelax_updated)))
    
    if save_slices:
        return results, slices

    return results


In [ ]:
# p = Parameters(Nx=6, Ny=6, Nz=6, a=2, sigma3=0.025, m3sq=-0.035, g3=-0.3, lambda3=1, rng=np.random.default_rng(seed=2))
# p = Parameters(Nx=6, Ny=6, Nz=6, a=1, sigma3=0.005, m3sq=-0.7, g3=0, lambda3=5, rng=np.random.default_rng(seed=5))


# this works okay at 4-5-6?
p = Parameters(Nx=5, Ny=5, Nz=5, a=1, sigma3=0.095, m3sq=-0.04, g3=-0.6, lambda3=1, rng=np.random.default_rng(seed=1))
# p = Parameters(Nx=6, Ny=6, Nz=6, a=1, sigma3=0.095, m3sq=-0.3, g3=-0.3, lambda3=1, rng=np.random.default_rng(seed=5))


# p = Parameters(Nx=6, Ny=6, Nz=6, a=1, sigma3=0, m3sq=-0.04, g3=-0.6, lambda3=1, rng=np.random.default_rng(seed=5))


lattice = init_lattice_warm(p,seed=10)

In [ ]:
x = np.linspace(-1.5,2.5,50)
plt.plot(x, local_potential(x, p))
plt.xlabel(r'$\phi$')
plt.ylabel(r'$V(\phi)$')

In [ ]:
results_all = []
slices_all = []

Running this many iterations takes a minute on my laptop (M4 MacBook Air), and about two minutes on mybinder.org:

In [ ]:
results, slices = run(5000, lattice, p, save_slices=True)

results_all += results
slices_all += slices

In [ ]:
phi = np.array(results_all)[:,0]
phisq = np.array(results_all)[:,1]
phi3 = np.array(results_all)[:,2]
action = np.array(results_all)[:,3]
pot = np.array(results_all)[:,4]/(p.Nx*p.Ny*p.Nz)

fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(figsize=(12,3), ncols=5)

ax1.plot(range(len(phi)),phi)
ax1.set_xlabel('measurement')
ax1.set_ylabel(r'$\overline{\phi}$')
ax2.plot(range(len(phisq)),phisq)
ax2.set_xlabel('measurement')
ax2.set_ylabel(r'$\overline{\phi^2}$')
ax3.plot(range(len(phi3)),phi3)
ax3.set_xlabel('measurement')
ax3.set_ylabel(r'$\overline{\phi^3}$')
ax4.plot(range(len(action)),action)
ax4.set_xlabel('measurement')
ax4.set_ylabel(r'$S$')
ax5.plot(range(len(pot)),pot)
ax5.set_xlabel('measurement')
ax5.set_ylabel(r'$\overline{V(\phi)}$')

plt.tight_layout()
plt.show()

Making this movie takes ??? on my MacBook M4, ??? seconds on mybinder.org.

In [ ]:

def plot_slices():
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10,3.5))

    x = np.linspace(-1.5,2.5,50)
    
    # Initial state, needed for colorbar
    pot_plot = ax1.plot(x, local_potential(x, p))
    ax1.set_xlabel(r'$\phi$ or $\overline{\phi}$')
    ax1.set_ylabel(r'$V(\phi)$ or $\overline{V(\phi)}$')
    sc_plot = ax1.scatter(phi[0], pot[0])
    lattice_slice = ax2.imshow(slices_all[0],vmin=-1.5,vmax=2.5)
    cbar = fig.colorbar(lattice_slice, ax=ax2, shrink=1.0)
    cbar.set_label(r'$\phi(\mathbf{x})$')
    plt.tight_layout()

    def update(frame):
        ax1.clear()
        ax2.clear()
        pot_plot = ax1.plot(x, local_potential(x, p))
        ax1.set_xlabel(r'$\phi$ or $\overline{\phi}$')
        ax1.set_ylabel(r'$V(\phi)$ or $\overline{V(\phi)}$')
        sc_plot = ax1.scatter(phi[frame], pot[frame])
        lattice_slice = ax2.imshow(slices_all[frame],vmin=-1.5,vmax=2.5)

    ani = FuncAnimation(fig, update, frames=range(0,len(slices_all),20), interval=100, repeat=False)
    return ani

movie = plot_slices().to_jshtml()
plt.close()
HTML(movie)


In [ ]:
(counts, edges, patches) = plt.hist(phi, bins=50)

In [ ]:
def reweight(dsigma=0, dmsq=0, dg=0):
    numerator = np.exp(-(dsigma*phi + (1.0/2.0)*dmsq*phisq + (1.0/6.0)*dg*phi3))
    denominator = np.sum(numerator)/len(results_all)
    weights = numerator/denominator
    return weights

In [ ]:
delta = np.arange(-6,6,0.01)
reweighted = []
for d in delta:
    reweighted.append(np.mean(phi*reweight(dmsq=d)))
plt.plot(delta, reweighted)
plt.xlabel(r'$\Delta = m^2 - m_0^2$')
plt.ylabel(r'$\langle \phi \rangle$')

In [ ]:
def plot_reweight(dsigma=0, dmsq=0, dg=0):
    weights = reweight(dsigma, dmsq, dg)
    plt.hist(phi, bins=50, weights=weights)
    plt.ylim(0,1000)
    plt.show()

ipywidgets.interact(plot_reweight, dsigma=(-4.0,4.0), dmsq=(-6.0,6.0), dg=(-5.0,5.0))

Things to try:

- **Easy**
  - Run for longer, and see what the peaks stabilise out at.
  - Decrease the lattice volume to say 4x4x4, and try increasing (if you have the time and patience to 8x8x8). How often does the simulation tunnel back and forth?
  - Explore (small) changes to the bare potential, how does it affect the distribution?
  - The overrelaxation is computationally costly. Is it worth it? Study how it works.

- **Medium**
  - Investigate different measurables and their histograms (phi, phi^2, the action, etc.). Do they all distinguish between the two phases?
  - Study autocorrelation times. How autocorrelated are these quantities. Do all the measurables have the same autocorrelation time?
  - Consider statistical errors. How many measurements does one have if one only considers say every N autocorrelation times an independent measurement? What about statistical bias - can one use jackknife or bootstrap?

- **Hard** [these will need you to run the code for a while, probably on your own laptop]
  - Add support for the multicanonical algorithm, and use it to build a weight function. The 'perfect' weight function tells you the free energy of the system immediately.
  - Implement the lattice-continuum relations described in https://arxiv.org/abs/2101.05528 and run the simulation to reproduce e.g. one of the curves in Figure 5.
